In [ ]:
from jaxtyping import Array, Float, Scalar, Key
import jax
import jax.numpy as jnp
import jax.random as jr
import equinox as eqx
import optax
from matplotlib import pyplot as plt
from tqdm import tqdm
jax.config.update("jax_enable_x64", True)

from src import lisa, networks
from wdm_transform.transforms import from_freq_to_wdm

In [ ]:
# problem
SEED = 0
N_SOURCES = 2
T_OBS = lisa.MONTH_s  # 1 month of observation

In [ ]:
u = jr.uniform(jr.key(SEED), shape=(N_SOURCES, 8))
params = lisa.prior_inverse_cdf(u)
signal = lisa.clean_signal(params, t_obs=T_OBS)
noise = lisa.sample_noise(jr.key(SEED+1), t_obs=T_OBS)
datastream = signal + noise
print(signal.shape, noise.shape)

n_samples = int(T_OBS / lisa.SAMPLING_STEP_s)
freqs = jnp.fft.rfftfreq(n_samples, lisa.SAMPLING_STEP_s)
n_freqs_crop = len(freqs) // 32 * 32
freqs_crop = freqs[:n_freqs_crop]

plt.figure(figsize=(12, 8))
for j, channel in enumerate("AET"):
    plt.subplot(311 + j)
    plt.title(f"channel {channel}")
    plt.loglog(freqs_crop, jnp.abs(datastream[:, j]), alpha=0.3, label="datastream")
    plt.loglog(freqs_crop, jnp.abs(signal[:, j]), alpha=0.3, label="signal")
    plt.xlim(params[:, 0].min(axis=0) * 0.9, params[:, 0].max(axis=0) * 1.1)
    plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# signal shape: (n_freqs_crop, 3) complex rfft coefficients for A/E/T
# transpose to (3, n_freqs_crop) so the batch axis is first
nt = 32
nf = len(signal) // nt  # 226 frequency bins
signal_wnm = from_freq_to_wdm(
    signal.T,
    nt=nt,
    nf=nf,
    a=1.0 / 3.0,
    d=1.0,
    dt=lisa.SAMPLING_STEP_s,
    backend="jax",
)
# shape: (3, nt, nf+1)  —  channels × time bins × frequency bins
print("signal_wnm shape:", signal_wnm.shape)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for j, ch in enumerate("AET"):
    im = axes[j].imshow(
        jnp.abs(signal_wnm[j]).T,  # (nf+1, nt)
        aspect="auto",
        origin="lower",
        interpolation="nearest",
        extent=[0, nt, 0, nf + 1]
    )
    axes[j].set_title(f"WDM |W|  —  channel {ch}")
    axes[j].set_xlabel("time bin n")
    axes[j].set_ylabel("frequency bin m")
    plt.colorbar(im, ax=axes[j])
plt.tight_layout()
plt.show()

In [ ]:
import einops
_, _, _, y = lisa.get_train_batch(
    jr.key(0), batch_size=10, n_sources=N_SOURCES, t_obs=T_OBS
)
y = einops.rearrange(y, "b t (f c) -> b c t f", c=3)

for i in range(10):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for j, ch in enumerate("AET"):
        im = axes[j].imshow(
            jnp.abs(y[i, j].T),  # (nf+1, nt)
            aspect="auto",
            origin="lower",
            interpolation="nearest",
            extent=[0, nt, 0, nf + 1]
        )
        axes[j].set_title(f"WDM |W|  —  channel {ch}")
        axes[j].set_xlabel("time bin n")
        axes[j].set_ylabel("frequency bin m")
        plt.colorbar(im, ax=axes[j])
    plt.tight_layout()
    plt.show()